# Branch and bound: the lecture 18 knapsack

Run the cells **from top to bottom**, pausing after each LP solve to ask what
should happen next. To repeat the demonstration, start again at the setup cell.
Open the course repository root in VS Code and select the **Julia 1.12**
kernel with the course environment. All required packages are included; no
external data files are needed. **Run All** also executes the complete demonstration.

We choose among four items:

| Item | 1 | 2 | 3 | 4 |
|:-----|--:|--:|--:|--:|
| Weight | 8 | 5 | 3 | 2 |
| Value | 15 | 12 | 4 | 2 |

$$
\begin{aligned}
\max\quad &15x_1+12x_2+4x_3+2x_4\\
\text{s.t.}\quad &8x_1+5x_2+3x_3+2x_4\le10,\\
&x_i\in\{0,1\},\qquad i=1,\ldots,4.
\end{aligned}
$$

This lecture example is adapted from MIT 15.053/8. We will build a five-node
tree by solving **continuous LP relaxations** and adding the branch
restrictions ourselves.

## Setup

The first helper builds a fresh LP relaxation, so branch restrictions from
one node cannot accidentally carry over to an unrelated node. The second
solves an LP and prints its bound and solution. The third reports global bounds.

In [ ]:
using JuMP, HiGHS
import MathOptInterface as MOI

function knapsack_relaxation()
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    @variable(model, 0 <= x[1:4] <= 1)
    @constraint(model, 8x[1] + 5x[2] + 3x[3] + 2x[4] <= 10)
    @objective(model, Max, 15x[1] + 12x[2] + 4x[3] + 2x[4])
    return model, x
end

function solve_node(model, x)
    optimize!(model)
    status = termination_status(model)
    println("LP status: ", status)
    if status == MOI.INFEASIBLE
        println("No feasible LP solution at this node.")
        return nothing
    end
    status == MOI.OPTIMAL || error("Unexpected solver status: $status")
    result = (bound=objective_value(model), x=value.(x))
    println("Node upper bound = ", round(result.bound; digits=6))
    println("LP solution x = ", round.(result.x; digits=6))
    return result
end

function show_bounds(L, active_bounds)
    U = maximum([L; collect(values(active_bounds))])
    println("Active nodes: ", sort(collect(keys(active_bounds))))
    for node in sort(collect(keys(active_bounds)))
        println("  Node ", node, " upper bound = ", round(active_bounds[node]; digits=6))
    end
    println("Global L = ", round(L; digits=6), ", U = ", round(U; digits=6),
            ", absolute gap = ", round(U - L; digits=6))
    return U
end

# The empty knapsack is our initial feasible solution.
incumbent = zeros(4)
L = 0.0
active_bounds = Dict(1 => Inf)
println("Initial incumbent: x = ", incumbent, ", value = ", L)

## Bounds for this maximization problem

- A feasible **binary** solution gives a global lower bound $L$.
- A node LP optimum gives an upper bound for **that subtree**.
- Each unsolved child inherits its parent's upper bound.
- The global bound is
  $$U=\max\bigl(\{L\}\cup\{U_i:i\text{ is active}\}\bigr).$$

The three reasons to close (or *prune*) a node are **integrality**,
**bound**, and **infeasibility**. We finish when no active nodes remain.

## Node 1: solve the root LP

We relax the binary requirements to $0\le x_i\le1$.
**Predict:** can the LP value be smaller than the best binary value?

In [ ]:
m, x = knapsack_relaxation()
node1 = solve_node(m, x)
active_bounds[1] = node1.bound
U = show_bounds(L, active_bounds)

The root solution is $(5/8,1,0,0)$ with value $171/8=21.375$.
It is fractional, and its bound exceeds $L$.

**Branch on $x_1$.** Node 2 requires $x_1=0$; node 3 requires $x_1=1$.
These two cases cover every binary solution. What bounds do the children
have before we solve their LPs?

In [ ]:
delete!(active_bounds, 1)
active_bounds[2] = node1.bound  # inherited bound
active_bounds[3] = node1.bound  # inherited bound
println("Branch: node 2 has x[1] = 0; node 3 has x[1] = 1.")
U = show_bounds(L, active_bounds)

## Node 2: $x_1=0$

Solve the left child. After seeing the result, decide whether to update
the incumbent, branch, or close this node.

In [ ]:
m, x = knapsack_relaxation()
@constraint(m, x[1] == 0)
node2 = solve_node(m, x)

**Pause:** Is this LP solution binary? What does that tell us about node 2?
Can we also close node 3 without solving it?

In [ ]:
# The optimal LP solution is binary and improves the incumbent.
incumbent = copy(node2.x)
L = node2.bound
delete!(active_bounds, 2)
println("Close node 2 by integrality; new incumbent = ", incumbent)
U = show_bounds(L, active_bounds)

Now $L=18$, but node 3 still has its inherited upper bound $171/8$.
Finding a good feasible solution has not yet proved it optimal.

## Node 3: $x_1=1$

Solve the other child of the root. Can its subtree improve on $L=18$?

In [ ]:
m, x = knapsack_relaxation()
@constraint(m, x[1] == 1)
node3 = solve_node(m, x)
active_bounds[3] = node3.bound
U = show_bounds(L, active_bounds)

The solution is $(1,2/5,0,0)$, with upper bound $99/5=19.8$.
It is fractional and the bound still exceeds $18$, so we branch on $x_2$.

Node 4 adds $x_2=0$ and node 5 adds $x_2=1$.
**Both retain the parent's restriction $x_1=1$.**

In [ ]:
delete!(active_bounds, 3)
active_bounds[4] = node3.bound
active_bounds[5] = node3.bound
println("Branch: node 4 has x[2] = 0; node 5 has x[2] = 1.")
U = show_bounds(L, active_bounds)

## Node 4: $x_1=1$, $x_2=0$

Remember to include **both** restrictions on the path from the root.

In [ ]:
m, x = knapsack_relaxation()
@constraint(m, x[1] == 1)
@constraint(m, x[2] == 0)
node4 = solve_node(m, x)

**Pause:** the solution is $(1,0,2/3,0)$, with upper bound
$53/3\approx17.6667$. It is fractional. Do we need to branch on $x_3$?

In [ ]:
if node4.bound <= L
    delete!(active_bounds, 4)
    println("Close node 4 by bound: even its LP optimum cannot improve on L = ", L)
else
    active_bounds[4] = node4.bound
    println("Node 4 cannot be closed by bound.")
end
U = show_bounds(L, active_bounds)

Node 5 remains active with inherited upper bound $19.8$.
Node 4's smaller LP value bounds only node 4's subtree, not the entire problem.

## Node 5: $x_1=1$, $x_2=1$

**Predict before solving:** these two items alone weigh $8+5=13$.
Can this node be feasible with capacity $10$?

In [ ]:
m, x = knapsack_relaxation()
@constraint(m, x[1] == 1)
@constraint(m, x[2] == 1)
node5 = solve_node(m, x)

**Pause:** which pruning test applies? Does infeasibility of this node
mean that the whole knapsack problem is infeasible?

In [ ]:
if node5 === nothing
    delete!(active_bounds, 5)
    println("Close node 5 by infeasibility.")
else
    active_bounds[5] = node5.bound
    println("Node 5 is feasible; further decisions are needed.")
end
U = show_bounds(L, active_bounds)

## Optimality certificate

No active nodes remain. The incumbent has value $18$, and every other
subtree has been closed. Thus $L=U=18$: the incumbent is globally optimal.

In [ ]:
isempty(active_bounds) || error("There are still active nodes.")
println("Optimal selection x = ", round.(Int, incumbent))
println("Total weight = ", sum([8, 5, 3, 2] .* incumbent))
println("Optimal value = ", L)
println("Certified bounds: L = ", L, ", U = ", U)

## The completed tree

```text
                  1: LP bound 21.375
                 /                  \
             x1 = 0                x1 = 1
               /                      \
     2: LP value 18                3: LP bound 19.8
     close: integral                /            \
     incumbent = 18             x2 = 0          x2 = 1
                                /                  \
                    4: LP bound 17.6667       5: infeasible
                    close: bound              close: infeasible
```

**Discussion:** Why was the incumbent found at node 2 useful at node 4?
What would change if we explored the right side of the tree first?